# Background Information

The customer support agents at a Telecom company use a RAG-based assistant to answer customer questions. While the assistant extracts relevant details from the company's internal support manuals, it sometimes misses details about new or emerging issues. In such cases, the agents refer to online community forums for the latest troubleshooting tips. However, the information on these forums is often unverfied, incomplete or contradictory.\

The task is to enhance the RAG system used by the customer support team by intergrating it with a dynamic data source. this will allow the team to access continuosly updated information instead of details extracted from static internal manuals.

## Task 1: Import all libraries

The following Pythin libraries are necessary for implementaing RAG system:
* sentence-transforms: provides pretrained models for generating vectors for sentences
* faiss-cpu: performs efficient similarity searcg and clustering of dense vectors
* langchain_community: provides intergrations with various tools and services, including LLM's
* replicate: Enables interaction with models hosted on the Replicate platform

In [ ]:
! pip install sentence-transformers faiss-cpu langchain_community replicate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from langchain_community.llms import Replicate
import os
from google.colab import userdata

/tmp/ipykernel_1761/1350509048.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Replicate


In [ ]:
api_token = userdata.get('Replicate_api_token')
os.environ["REPLICATE_API_TOKEN"] = api_token
model = "ibm-granite/granite-3.2-8b-instruct"
output = Replicate(
     model=model,
     replicate_api_token=api_token,
     )


## Task 2: Extract content from dynamic knowledge sources through web crawling

In this task, you'll integrate a dynamic knowledge base into your RAG system. You'll gather content using the web crawling technique. then, you'll chunk it, create embeddings for the chunk content, and store the chunks in a Facebook AI Similarity Search(FAISS) database.

FAISS is an open-source library that acts as the RAG system's search engine. It helps RAG quickly find the most releavnt information from a large pool of vectorized data.

Web crawling is the automated process of systematically c=browsing the wen to discover, analyze and extract information from websites.

Web crawling is a process often used to enable a RAG system to:
* Automate the collection of website content for the system
* Keep training data up-to-date through scheduled crawls
* Expand source civerage by crawling linked pages wthin a domain or across related domains
* Improve retreival accuract with frest, indexed content



Developers often use test data when testing updated made to a system. In this case, you are using thealliance.ai as the test data to validate that the system can connect to external web sources and retrieve text data successfully.

The Alliance AI is a non-profit publicly available community focused on open innovation in AI. This site regulary published new content on AI, making it ideal for testing how your RAG system handles live data

To produce relevant output, a RAG system needs a corpus of clean, text-based data. The code in this step extracts information from thealliance.ai website and saves it to the text file to create this corpus. this code is used to download the webpage's content, identify the readable text, clean it and finally, save it in a file for later use.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

url = "https://thealliance.ai/"
global page_text
page_text = "" # Initialize page_text

try:
  resp = requests.get(url, timeout=15)
  resp.raise_for_status()
  soup = BeautifulSoup(resp.text, "html.parser")
  # Remove script/style/noscript tags then get visible text
  for s in soup(["script", "style", "noscript"]):
    s.decompose()
  page_text = soup.get_text(separator="\n")

  # collapse multiple blank lines
  page_text = re.sub(r'\n\s*\n+', '\n\n', page_text)

  # Optionally save to file
  with open("thealliance_ai_content.txt", "w", encoding="utf-8") as f:
    f.write(page_text)
  print(f"Crawled {url} -> saved to 'thealliance_ai_content.txt' (chars: {len(page_text)})")
except Exception as e:
  print("Web crawl failed. You can create `page_text` manually. Error:", e)

Crawled https://thealliance.ai/ -> saved to 'thealliance_ai_content.txt' (chars: 6066)


Chunking the data into manageable and sematically meaningful pieces

In [ ]:
# Create chunks for web crawl
def split_into_chunks(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Always run if page_text exists
if 'page_text' in globals():
    chunks_web = split_into_chunks(page_text)
    print(f"Web-crawled content split into {len(chunks_web)} chunks.")
else:
    raise RuntimeError("No web-crawled text found. Run the crawl first.")

Web-crawled content split into 25 chunks.


This code converts each chunk into an embedding using a sentence-transformer model. it also store the embeddings in a searchable FAISS database

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
# Ensure embedding model (will reuse if already present)
try:
    embed_model
except NameError:
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    print("Loaded embedding model 'all-MiniLM-L6-v2'")
def build_index_from_chunks(chunks_local):
    embs = embed_model.encode(chunks_local, convert_to_numpy=True).astype(np.float32)
    dim = embs.shape[1]
    idx = faiss.IndexFlatL2(dim)
    idx.add(embs)
    return idx, embs
# Build web index if needed
if 'chunks_web' in globals():
    if 'index_web' not in globals():
        index_web, embeddings_web = build_index_from_chunks(chunks_web)
        chunk_store_web = chunks_web
        print(f"Built FAISS index for web source ({len(chunks_web)} chunks).")
    else:
        print("Web FAISS index already present.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model 'all-MiniLM-L6-v2'
Built FAISS index for web source (25 chunks).


## Task 3: Test the system to access the response quality

Check how well your RAG system retreives and generated answers in repsonse to a query

In [ ]:
try:
    user_query
except NameError:
    user_query = input("Enter your query: ")
print("The query you entered: ", user_query)

Enter your query: Tell more about AI Alliance
The query you entered:  Tell more about AI Alliance


Retrieve only the top 3 results. k=3

In [ ]:
import numpy as np
def retrieve_top_k(index_obj, chunk_store_local, query, k=3):
    q_emb = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, ids = index_obj.search(q_emb, k)
    top_chunks = [chunk_store_local[i] for i in ids[0]]
    return top_chunks, distances[0], ids[0]
retrieval_results = {}
if 'index_web' in globals() and 'chunk_store_web' in globals():
    top_chunks_web, dist_web, ids_web = retrieve_top_k(index_web, chunk_store_web, user_query, k=3)
    retrieval_results['web'] = {
        'top_chunks': top_chunks_web,
        'distances': dist_web,
        'ids': ids_web
    }
    print(f"Web: retrieved {len(top_chunks_web)} chunks.")
if not retrieval_results:
    raise RuntimeError("No index found. Run Step 7 to build indexes from chunks.")

Web: retrieved 3 chunks.


Display the Top 3 results

In [ ]:
for source in ['web']:
    if source in retrieval_results:
        print("\n" + "="*8 + f" Retrieved chunks — {source.upper()} " + "="*8 + "\n")
        for i, ch in enumerate(retrieval_results[source]['top_chunks'], start=1):
            print(f"[{i}] {ch[:500].replace('\n',' ')}\n")


======== Retrieved chunks — WEB ========

[1]   AI Alliance  About  About | AI Alliance | Leadership  Projects  Events  Join  Blog  Donate  AI Alliance  A global network advancing open, safe, and responsible AI through open innovation, collaboration, and advocacy  Home  AI Alliance  To accelerate open innovation in AI, advancing core capabiliti

[2] ve always been true catalysts for progress. The launch of the AI Alliance marks a visionary milestone, uniting industry giants, academia, and innovators with a shared responsibility to shape the future and advance open innovation, ensuring that the transformative power of AI is harnessed responsibly

[3] edge systems, helping developers build smarter workflows, retain insights, and create powerful, personalized knowledge bases.  									Sign up for the in person event  1  /  1  The AI Alliance is a collaborative network of companies, startups, universities, research institutions, government organiz

